<a href="https://colab.research.google.com/github/ShilpaAjitheks/LLM/blob/main/Bert_MC_LLM_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [ ]:
!pip install datasets

In [ ]:
!pip3 install accelerate -U

In [ ]:
!pip install transformers[torch]

In [ ]:
!pip install transformers[sentencepiece]

In [ ]:
!pip install tensorflow
!pip install --upgrade pip
!pip install tensorflow_hub
import tensorflow as tf
import tensorflow_hub as hub
!pip install tensorflow-text
# !pip install tensorflow_hub
# import tensorflow_text as text

2023-10-27 08:43:13.913349: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-27 08:43:14.114139: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-10-27 08:43:14.114168: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-10-27 08:43:14.114995: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-10-27 08:43:14.215242: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-10-27 08:43:14.217936: I tensorflow/core/platform/cpu_feature_guard.cc:182] This Tens

In [ ]:
!pip install evaluate

In [ ]:
!pip install tensorflow-text
# import tensorflow_text as text

In [ ]:
# import os
# from google.colab import drive
# drive.mount('/content/drive',force_remount=True)
# os.chdir("drive/My Drive/Colab Notebooks")

In [ ]:
!pip install scikit-learn
!pip install openpyxl

# Importing Libraries

In [ ]:
from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
import pandas as pd
from datasets import Dataset
from tqdm.auto import tqdm
import evaluate
from sklearn.metrics import f1_score, classification_report, precision_score, recall_score

import torch
from transformers import AdamW, AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from transformers import TrainingArguments
from transformers import AutoModelForSequenceClassification
from transformers import get_scheduler
from transformers import AdamW
import requests
import ast
import time
import requests
import urllib
import numpy as np
import shutil
import pickle
from torch.optim import Adagrad,Adadelta,RMSprop
from torch.optim.lr_scheduler import CosineAnnealingLR,PolynomialLR,ExponentialLR

# API String

In [ ]:
LLM_DATA_LOADING = 'http://0.0.0.0:4000/mc_llm_data_loading'
api_token = 'xyz'
workflow_id = "302"
STATUS_URL="http://0.0.0.0:4000/prediction_results/"
FILE_UPLOAD_URL = 'http://0.0.0.0:4000/llm_model_upload'
LLM_UPDATE_TRAIN_INFO = 'http://0.0.0.0:4000/mc_llm_training_info_update'

## Functions

In [ ]:
#Data Loading
def get_data_link():
    response = (requests.post(LLM_DATA_LOADING, headers={"x-access-token": api_token}, json={"workflowId": workflow_id})).text

    while True:
        time.sleep(10)
        status = ((requests.get(STATUS_URL+response, headers={"x-access-token": api_token})).json())
        if status['state'] == 'SUCCESS':
            if 'failed' in status['result']:
                print("Stage1 failed")
                return False
            else:
                break

    return(status)

def download_data(status):
    filenames = []
    for key,value in status['predicted_result'].items():
        url = value
        print(value)
        filename = url.split('/')[-1]
        urllib.request.urlretrieve(url, filename)
        filenames.append(filename)

    return(filenames)



def filter_active_classes(training_data_df,masterlist_classes):
    all_classes = training_data_df['Class Name'].values
    print(len(all_classes))
    # all_classes = [ast.literal_eval(clas) for clas in all_classes]
    # print(all_classes)
    valid_indexes = []
    for i in range(len(all_classes)):
        if all_classes[i][0] in masterlist_classes:
            valid_indexes.append(i)

    valid_train_data = training_data_df.iloc[valid_indexes]

    return(valid_train_data)

def data_loading():
    print('data loading')
    #DataLoading from DataNeuron API
    status = get_data_link()
    print('got link')
    model_params = status['predicted_result']['model_params']
    masterlist_classes = status['predicted_result']['masterlist_nodes']
    del status['predicted_result']['model_params']
    del status['predicted_result']['masterlist_nodes']
    filenames = download_data(status)
    print('data downloaded')
    training_data_df = pd.read_excel(filenames[0])
    unverified_data_df = pd.read_excel(filenames[1])

    #Data filter for active classes
    training_data_df = training_data_df.dropna()
    all_classes = training_data_df['Class Name'].values
    # print(all_classes)
    training_data_df['Class Name'] = [ast.literal_eval(clas) for clas in all_classes]
    valid_train_data = filter_active_classes(training_data_df,masterlist_classes)
    print('filtered data')
    print(valid_train_data['Class Name'])
    valid_train_data['Class Name'] = valid_train_data['Class Name'].apply(lambda x: x[0])

    #Data Prep
    x = valid_train_data['Paragraph Text'].values
    x = np.array([value for value in x])
    x = np.reshape(x, (x.shape[0], -1))
    lb = LabelBinarizer()
    y = lb.fit_transform(valid_train_data['Class Name'])

    # model_params['num_classes'] = len(valid_train_data['Class Name'].unique())
    model_params['num_classes'] = y.shape[1]

    # Iterative stratification for multi-label data
    (x_train,x_test,y_train, y_test) = train_test_split(x, y,stratify=y,test_size = model_params['test_size'])

    print(x_train.shape)
    print(x_test.shape)
    print(y_train.shape)
    print(y_test.shape)
    print(model_params)
    # print(x_test)

    return(x_train,x_test,y_train,y_test,lb,model_params,unverified_data_df)




def data_preparation(x_train,x_test,y_train,y_test,model_params):
    print('data preparation')
    checkpoint = "bert-base-cased"
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    x_train = [item for sublist in x_train for item in sublist]
    x_test = [item for sublist in x_test for item in sublist]

    # data_size = 50
    train={}
    train['idx'] = [i for i in range(len(x_train))]
    train['sentence'] = x_train
    train['label'] = y_train

    val = {}
    val['idx'] = [i for i in range(len(x_test))]
    val['sentence'] = x_test
    val['label'] = y_test

    # max_target_length = 50
    train['input_ids'] = []
    train['token_type_ids'] = []
    train['attention_mask'] = []
    for i in range(len(train['sentence'])):
        data = train['sentence'][i]
        tokenized_data = tokenizer(data, truncation=True,padding=True)
        train['input_ids'].append(tokenized_data['input_ids'])
        train['token_type_ids'].append(tokenized_data['token_type_ids'])
        train['attention_mask'].append(tokenized_data['attention_mask'])

    del train['sentence']
    del train['idx']

    val['input_ids'] = []
    val['token_type_ids'] = []
    val['attention_mask'] = []
    for i in range(len(val['sentence'])):
        data = val['sentence'][i]
        # print(data)
        tokenized_data = tokenizer(data, truncation=True,padding=True)
        val['input_ids'].append(tokenized_data['input_ids'])
        val['token_type_ids'].append(tokenized_data['token_type_ids'])
        val['attention_mask'].append(tokenized_data['attention_mask'])

    del val['sentence']
    del val['idx']


    print('train : ',len(train['input_ids']))
    print('val : ',len(val['input_ids']))

    train_dataset = Dataset.from_dict(train)
    val_dataset = Dataset.from_dict(val)

    dataset = {"train":train_dataset,"validation":val_dataset,"test":val_dataset}


    train_dataloader = DataLoader(
        dataset["train"], batch_size=model_params['batch_size'], collate_fn=data_collator
    )
    eval_dataloader = DataLoader(
        dataset["validation"], batch_size=model_params['batch_size'], collate_fn=data_collator
    )

    return(train_dataloader,eval_dataloader)

def model_setup(train_dataloader,model_params):
    checkpoint = "bert-base-cased"
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=model_params['num_classes'])

    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    model.to(device)
    # model.to('cuda')
    # optimizer = AdamW(model.parameters(), lr=5e-5)

    selected_optimizer = model_params['optimizer']

    if selected_optimizer == 'AdamW':
        optimizer = AdamW(model.parameters(), lr=float(model_params['learning_rate']))
    elif selected_optimizer == 'RMSprop':
        optimizer = RMSprop(model.parameters(), lr=float(model_params['learning_rate']))
    elif selected_optimizer == 'Adagrad':
        optimizer = Adagrad(model.parameters(), lr=float(model_params['learning_rate']))
    elif selected_optimizer == 'Adadelta':
        optimizer = Adadelta(model.parameters(), lr=float(model_params['learning_rate']))
    else:
        raise ValueError(f"Unsupported optimizer: {selected_optimizer}")

    num_epochs = model_params['epochs']
    num_training_steps = num_epochs * len(train_dataloader)
    # lr_scheduler = get_scheduler(
    #     "linear",
    #     optimizer=optimizer,
    #     num_warmup_steps=0,
    #     num_training_steps=num_training_steps,
    # )

    selected_scheduler = model_params['lr_scheduler']

    if selected_scheduler == 'linear':
        lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=4, num_training_steps=num_training_steps)
    elif selected_scheduler == 'cosine':
        lr_scheduler = CosineAnnealingLR(optimizer=optimizer, T_max=num_training_steps, eta_min=0, last_epoch=-1, verbose=True)
    elif selected_scheduler == 'polynomial':
        lr_scheduler = PolynomialLR(optimizer=optimizer, total_iters=num_training_steps, power=2, last_epoch=-1, verbose=True)
    elif selected_scheduler == 'exponential':
        lr_scheduler = ExponentialLR(optimizer=optimizer, gamma=0.1, last_epoch=-1, verbose=False)
    else:
        raise ValueError(f"Unsupported scheduler: {selected_scheduler}")

    # print(num_training_steps)
    return(model,optimizer,lr_scheduler,num_training_steps)


def model_training(model,optimizer,lr_scheduler,num_training_steps,train_dataloader,model_params):
    print('model training')
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    progress_bar = tqdm(range(num_training_steps))
    num_epochs=model_params['epochs']
    model.train()
    for epoch in range(num_epochs):
        for batch in train_dataloader:
            # print('1')
            batch = {k: v.to(device) for k, v in batch.items()}
            # print('2')
            outputs = model(**batch)
            # print('3')
            loss = outputs.loss
            # print(loss)
            # print('3.1')
            loss.backward()
            # print('3.2')
            optimizer.step()
            # print('4')
            lr_scheduler.step()
            # print('5')
            optimizer.zero_grad()
            # print('6')
            progress_bar.update(1)

    return(model)



def model_evaluation(model,eval_dataloader,lb,model_params):
    print('model evaluation')
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
    metric = evaluate.load("f1",average='micro')
    model.eval()
    all_prediction = []
    all_true = []
    for batch in eval_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        # return(predictions)
        # print('predictions : ',predictions)
        predictions_lb = lb.transform(predictions.cpu().data.numpy())
        predictions_lb = torch.Tensor(predictions_lb.tolist())
        predictions_lb = predictions_lb.to(device)
        # print('predictions_lb : ',predictions_lb)
        # print('predictions shape : ',predictions_lb.shape)
        # print('batch shape : ',batch["labels"].shape)
        # metric.add_batch(predictions=predictions_lb.cpu().data.numpy(), references=batch["labels"].cpu().data.numpy())
        all_prediction.extend(predictions_lb.cpu().data.numpy())
        all_true.extend(batch["labels"].cpu().data.numpy())

    cr = classification_report(all_true, all_prediction, zero_division = True, output_dict = False)
    print('cr : ',cr)
    # print(metric.compute(average='micro'))
    return(cr)


def send_chunks(url,filename, chunk_size=4*1024*1024):  # 4MB chunks
    i=0
    rqp = '?is_last=false&filename='+filename+'&workflow_id='+workflow_id
    normal_url = url+rqp
    with open(filename, 'rb') as file:
        while True:
            data_chunk = file.read(chunk_size)
            if not data_chunk:
                rqp_last = '?is_last=true&filename='+filename+'&workflow_id='+workflow_id
                last_url = url+rqp_last
                response = requests.post(last_url, data=None, headers={"x-access-token": api_token})
                break
            response = requests.post(normal_url, data=data_chunk, headers={"x-access-token": api_token})
            print(response.text,' : ',i)
            i+=1

def data_preparation_ingested_data(unverified_data_df,model_params):
    print('data preparation')
    checkpoint = "bert-base-cased"
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    paragraphs = unverified_data_df['Paragraph Text']
    para_id = unverified_data_df['paraId']
    unverified = {}
    unverified['idx'] = [pid for pid in para_id]
    unverified['sentence'] = paragraphs

    unverified['input_ids'] = []
    unverified['token_type_ids'] = []
    unverified['attention_mask'] = []
    for i in range(len(unverified['sentence'])):
        data = unverified['sentence'][i]
        tokenized_data = tokenizer(data[0], truncation=True,padding=True)
        unverified['input_ids'].append(tokenized_data['input_ids'])
        unverified['token_type_ids'].append(tokenized_data['token_type_ids'])
        unverified['attention_mask'].append(tokenized_data['attention_mask'])

    del unverified['sentence']
    del unverified['idx']

    print('unverified : ',len(unverified['input_ids']))
    unverified_dataset = Dataset.from_dict(unverified)
    dataset = {"unverified":unverified_dataset}

    unverified_dataloader = DataLoader(
        dataset["unverified"], batch_size=model_params['batch_size'], collate_fn=data_collator
    )

    return(unverified_dataloader)

def prediction_on_ingested_data(model, unverified_dataloader,lb, model_params):
    print('Model evaluation')
    device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")


    metric = evaluate.load("f1",average='micro')

    model.eval()
    all_predictions = []

    for batch in unverified_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=1)  # Select the class with the highest probability
        predictions = predictions.cpu().data.numpy()

        all_predictions.extend(predictions)

    return all_predictions



def update_llm_training_info(workflow_id):
    response = (requests.post(LLM_UPDATE_TRAIN_INFO, headers={"x-access-token": api_token}, json={"workflow_id": workflow_id})).text

    while True:
        time.sleep(10)
        status = ((requests.get(STATUS_URL+response, headers={"x-access-token": api_token})).json())
        if status['state'] == 'SUCCESS':
            if 'failed' in status['result']:
                print("Stage1 failed")
                return False
            else:
                break

    return(status)

def main_Multiclass_LLM_BERT(workflow_id_param,api_token_param,LLM_DATA_LOADING_PARAM,STATUS_URL_PARAM,FILE_UPLOAD_URL_PARAM,LLM_UPDATE_TRAIN_INFO_PARAM):
    global workflow_id
    global api_token
    global LLM_DATA_LOADING
    global STATUS_URL
    global FILE_UPLOAD_URL
    global LLM_UPDATE_TRAIN_INFO

    workflow_id = workflow_id_param
    api_token = api_token_param
    LLM_DATA_LOADING = LLM_DATA_LOADING_PARAM
    STATUS_URL = STATUS_URL_PARAM
    FILE_UPLOAD_URL = FILE_UPLOAD_URL_PARAM

    print('1. Started')
    x_train,x_test,y_train,y_test,lb = data_loading()
    print('2. Data Loading Success')
    y_train = y_train.astype(float)
    y_test = y_test.astype(float)
    print(y_test.shape[1])
    train_dataloader,eval_dataloader = data_preparation(x_train,x_test,y_train,y_test)
    print('3. Data Preparation Success')
    model,optimizer,lr_scheduler,num_training_steps = model_setup(train_dataloader)
    print('4. Model Setup Success')
    num_epochs = 1
    model = model_training(model,optimizer,lr_scheduler,num_training_steps,train_dataloader)
    print('5. Model Training Success')
    cr = model_evaluation(model,eval_dataloader,lb)
    print('6. Model EvaluationSuccess')
    # print('classification result : ',cr)

    dirname = 'Multiclass_Bert_'+workflow_id
    model.save_pretrained(dirname)
    shutil.make_archive(dirname, 'zip', dirname)
    with open('Classification_Report_'+workflow_id+'.pkl','wb') as f:
      pickle.dump(cr,f)
    print('7. Model And Report Saved')

    try:
        send_chunks(FILE_UPLOAD_URL,filename = dirname+'.zip')
    except:
        print('Model Upload Fail')
    try:
        send_chunks(FILE_UPLOAD_URL,filename = 'Classification_Report_'+workflow_id+'.pkl')
    except:
        print('Classification Report Upload Fail')

    print('8. Model And Report Uploaded')

    try:
        if model_params['prediction_ingested_data']=='True':
            unverified_dataloader = data_preparation_ingested_data(unverified_data_df,model_params)
            print('9. unverified_dataloader created')
            unverified_results = prediction_on_ingested_data(model,unverified_dataloader,lb,model_params)
            print('10. unverified paragraphs prediction done')
            with open('Predicton_Result_'+workflow_id+'.pkl','wb') as f:
                pickle.dump(unverified_results,f)
            send_chunks(FILE_UPLOAD_URL,filename = 'Predicton_Result_'+workflow_id+'.pkl')
            print('11. unverified paragraphs uploaded')

    except:
        print('Prediction on Unverified Paras Fail')

    update_llm_training_info(workflow_id)


# Calling

In [ ]:
# if __name__=='__main__':
#     main_Multiclass_LLM_BERT()

In [ ]:
#multiclass data testing without model upload
print('1. Started')
x_train,x_test,y_train,y_test,mlb,model_params,unverified_data_df = data_loading()
print('2. Data Loading Success')
# print(y_train)
y_train = y_train.astype(float)
y_test = y_test.astype(float)
train_dataloader,eval_dataloader = data_preparation(x_train,x_test,y_train,y_test,model_params)
print('3. Data Preparation Success')
model,optimizer,lr_scheduler,num_training_steps = model_setup(train_dataloader,model_params)
print('4. Model Setup Success')
model = model_training(model,optimizer,lr_scheduler,num_training_steps,train_dataloader,model_params)
print('5. Model Training Success')
cr = model_evaluation(model,eval_dataloader,mlb,model_params)
print('6. model_evaluation Success')

#

1. Started
data loading
got link
https://dsalpartifacts.blob.core.windows.net/exports/302c5c6a-c90f-4625-bedc-f483d47f29df_training_dataset.xlsx
https://dsalpartifacts.blob.core.windows.net/exports/302c5c6a-c90f-4625-bedc-f483d47f29df_unverified_dataset.xlsx
data downloaded
125
filtered data
0          [earn]
1          [earn]
2          [earn]
3          [earn]
4          [earn]
          ...    
120       [grain]
121       [crude]
122       [crude]
123       [crude]
124    [interest]
Name: Class Name, Length: 125, dtype: object
(100, 1)
(25, 1)
(100, 7)
(25, 7)
{'batch_size': 8, 'epochs': 10, 'learning_rate': 0.001, 'lr_scheduler': 'linear', 'optimizer': 'AdamW', 'prediction_ingested_data': 'True', 'test_size': 0.2, 'userSelectedModel': 'Bert-Based-Uncased', 'num_classes': 7}
2. Data Loading Success
data preparation
train :  100
val :  25
3. Data Preparation Success


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


4. Model Setup Success
model training


/home/shilpa/jupyter/jupyter_notebook/lib/python3.10/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  0%|          | 0/130 [00:00<?, ?it/s]

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


5. Model Training Success
model evaluation


cr :                precision    recall  f1-score   support

           0       1.00      0.00      0.00         4
           1       1.00      0.00      0.00         4
           2       1.00      0.00      0.00         4
           3       1.00      0.00      0.00         4
           4       1.00      0.00      0.00         3
           5       1.00      0.00      0.00         3
           6       1.00      0.00      0.00         3

   micro avg       1.00      0.00      0.00        25
   macro avg       1.00      0.00      0.00        25
weighted avg       1.00      0.00      0.00        25
 samples avg       1.00      0.00      0.00        25

6. model_evaluation Success


In [ ]:
dirname = 'Multiclass_Bert_'+workflow_id
model.save_pretrained(dirname)
shutil.make_archive(dirname, 'zip', dirname)
with open('Classification_Report_'+workflow_id+'.pkl','wb') as f:
  pickle.dump(cr,f)
print('7. Model And Report Saved')
send_chunks(FILE_UPLOAD_URL,filename = dirname +'.zip')
send_chunks(FILE_UPLOAD_URL,filename = 'Classification_Report_'+workflow_id+'.pkl')
print('8. Model And Report Uploaded')

if model_params['prediction_ingested_data']=='True':
    unverified_dataloader = data_preparation_ingested_data(unverified_data_df,model_params)
    print('9. unverified_dataloader created')
    unverified_results = prediction_on_ingested_data(model,unverified_dataloader,lb,model_params)
    print('10. unverified paragraphs prediction done')
    with open('Predicton_Result_'+workflow_id+'.pkl','wb') as f:
        pickle.dump(unverified_results,f)
    send_chunks(FILE_UPLOAD_URL,filename = 'Predicton_Result_'+workflow_id+'.pkl')
    print('11. unverified paragraphs uploaded')

7. Model And Report Saved
5b68459a-884e-43f9-a928-43ad43ac3e24  :  0
b1e948e8-7302-40d7-ba85-2be15f3ea7fd  :  1
d6508c4b-6f84-46ff-bb09-7897db308ffc  :  2
246ccee7-4d60-43b0-abe0-ece7e29661ae  :  3
fdab792f-2829-424d-aafd-ac7dce77f40b  :  4
9a5cf662-cdb3-44ef-abc8-ec9b3e07cbe0  :  5
a9e09c50-85a1-46a0-a4bb-094d13605a18  :  6
88c02370-cc1e-498e-8709-4a0f4c5c243e  :  7
7f1586af-177a-48f1-9989-415799dc8ed4  :  8
6d625cc5-b7b6-4381-8aea-63bc9c2fb608  :  9
a88c3847-3c3a-483d-b1fa-c92cbcc5ffd4  :  10
136e6916-962d-44ff-a8fa-e9c6a19a73b3  :  11
af666931-b2c3-49a5-8715-1fea240df611  :  12
8dbe75e3-1611-468b-abff-6c7ef891216b  :  13
825fc5c1-54f7-4ce6-b504-e597a4a63950  :  14
